In [1]:
import pandas as pd

incident = pd.read_csv("data/NIBRS_incident.csv")
offense = pd.read_csv("data/NIBRS_OFFENSE.csv")
offense_type = pd.read_csv("data/NIBRS_OFFENSE_TYPE.csv")
location_type = pd.read_csv("data/NIBRS_LOCATION_TYPE.csv")
agencies = pd.read_csv("data/agencies.csv")

In [2]:
# Confirm the data I need that is linked to the Chicago dataset.
print("incident:", incident.shape)
print(incident.columns.tolist(), "\n")

print("offense:", offense.shape)
print(offense.columns.tolist(), "\n")

print("offense_type:", offense_type.shape)
print(offense_type.columns.tolist(), "\n")

print("location_type:", location_type.shape)
print(location_type.columns.tolist(), "\n")

print("agencies:", agencies.shape)
print(agencies.columns.tolist(), "\n")

incident: (1464023, 15)
['data_year', 'agency_id', 'incident_id', 'nibrs_month_id', 'cargo_theft_flag', 'submission_date', 'incident_date', 'report_date_flag', 'incident_hour', 'cleared_except_id', 'cleared_except_date', 'incident_status', 'data_home', 'orig_format', 'did'] 

offense: (1601693, 8)
['data_year', 'offense_id', 'incident_id', 'offense_code', 'attempt_complete_flag', 'location_id', 'num_premises_entered', 'method_entry_code'] 

offense_type: (86, 8)
['offense_code', 'offense_name', 'crime_against', 'ct_flag', 'hc_flag', 'hc_code', 'offense_category_name', 'offense_group'] 

location_type: (47, 3)
['location_id', 'location_code', 'location_name'] 

agencies: (1532, 59)
['yearly_agency_id', 'agency_id', 'data_year', 'ori', 'legacy_ori', 'covered_by_legacy_ori', 'direct_contributor_flag', 'dormant_flag', 'dormant_year', 'reporting_type', 'ucr_agency_name', 'ncic_agency_name', 'pub_agency_name', 'pub_agency_unit', 'agency_status', 'state_id', 'state_name', 'state_abbr', 'state

In [3]:
# perform a small-scale merge to facilitate my inspection of table structures and field quality.
tx_master = incident.merge(
    offense,
    on=["incident_id", "data_year"],
    how="inner"
)

tx_master = tx_master.merge(
    offense_type,
    on="offense_code",
    how="left"
)

tx_master = tx_master.merge(
    location_type,
    on="location_id",
    how="left"
)

tx_master = tx_master.merge(
    agencies,
    on=["agency_id", "data_year"],
    how="left"
)

print(tx_master.shape)
print(tx_master[["incident_id", "agency_id", "incident_date", "offense_code", "offense_name", "location_name", "county_name"]].isna().sum())
print(tx_master[["offense_name", "location_name", "county_name"]].head(10))

(1601693, 87)
incident_id       0
agency_id         0
incident_date     0
offense_code      0
offense_name      0
location_name     0
county_name      10
dtype: int64
                              offense_name  \
0                       Aggravated Assault   
1                           Simple Assault   
2                    Weapon Law Violations   
3                 Theft From Motor Vehicle   
4  False Pretenses/Swindle/Confidence Game   
5             Burglary/Breaking & Entering   
6                 Theft From Motor Vehicle   
7                        All Other Larceny   
8                 Drug/Narcotic Violations   
9                           Simple Assault   

                        location_name county_name  
0                      Residence/Home    ANDERSON  
1                      Residence/Home    ANDERSON  
2                      Residence/Home    ANDERSON  
3             Parking/Drop Lot/Garage    ANDERSON  
4               Bank/Savings and Loan    ANDERSON  
5             

In [6]:
# Review the distribution of high-frequency crimes and the complete crime list to prepare for the subsequent Texas → Chicago label mapping.

In [4]:
print(tx_master["offense_name"].value_counts().head(30))

offense_name
Simple Assault                                 257073
All Other Larceny                              172438
Destruction/Damage/Vandalism of Property       164903
Drug/Narcotic Violations                       140875
Theft From Motor Vehicle                       132181
Motor Vehicle Theft                            102607
Shoplifting                                     96299
Burglary/Breaking & Entering                    83813
Intimidation                                    74278
Aggravated Assault                              70272
Drug Equipment Violations                       42159
False Pretenses/Swindle/Confidence Game         33330
Weapon Law Violations                           31667
Theft of Motor Vehicle Parts or Accessories     30136
Credit Card/Automated Teller Machine Fraud      25268
Robbery                                         19926
Identity Theft                                  19549
Theft From Building                             18860
Counterfeiting/

In [5]:
all_offenses = sorted(tx_master["offense_name"].dropna().unique().tolist())
print(all_offenses)
print("Number of unique offenses:", len(all_offenses))

['Aggravated Assault', 'All Other Larceny', 'Animal Cruelty', 'Arson', 'Assisting or Promoting Prostitution', 'Betting/Wagering', 'Bribery', 'Burglary/Breaking & Entering', 'Counterfeiting/Forgery', 'Credit Card/Automated Teller Machine Fraud', 'Criminal Sexual Contact', 'Destruction/Damage/Vandalism of Property', 'Drug Equipment Violations', 'Drug/Narcotic Violations', 'Embezzlement', 'Extortion/Blackmail', 'False Pretenses/Swindle/Confidence Game', 'Gambling Equipment Violation', 'Hacking/Computer Invasion', 'Human Trafficking, Commercial Sex Acts', 'Human Trafficking, Involuntary Servitude', 'Identity Theft', 'Impersonation', 'Incest', 'Intimidation', 'Justifiable Homicide', 'Kidnapping/Abduction', 'Motor Vehicle Theft', 'Murder and Nonnegligent Manslaughter', 'Negligent Manslaughter', 'Operating/Promoting/Assisting Gambling', 'Pocket-picking', 'Pornography/Obscene Material', 'Prostitution', 'Purchasing Prostitution', 'Purse-snatching', 'Rape', 'Robbery', 'Sexual Assault With An Obj

In [7]:
# To align with the Chicago dataset, subcategories of crimes are grouped under these top three major crime types.
theft_set = {
    "All Other Larceny",
    "Shoplifting",
    "Theft From Building",
    "Theft From Coin-Operated Machine or Device",
    "Theft From Motor Vehicle",
    "Theft of Motor Vehicle Parts or Accessories",
    "Pocket-picking",
    "Purse-snatching"
}

battery_set = {
    "Simple Assault",
    "Aggravated Assault"
}

criminal_damage_set = {
    "Destruction/Damage/Vandalism of Property"
}

In [8]:
def map_crime_category(offense_name):
    if offense_name in theft_set:
        return "THEFT"
    elif offense_name in battery_set:
        return "BATTERY"
    elif offense_name in criminal_damage_set:
        return "CRIMINAL_DAMAGE"
    else:
        return "OTHER"

In [9]:
tx_master["crime_category_mapped"] = tx_master["offense_name"].apply(map_crime_category)

In [10]:
tx_master["is_total_crime"] = 1
tx_master["is_theft"] = (tx_master["crime_category_mapped"] == "THEFT").astype(int)
tx_master["is_battery"] = (tx_master["crime_category_mapped"] == "BATTERY").astype(int)
tx_master["is_criminal_damage"] = (tx_master["crime_category_mapped"] == "CRIMINAL_DAMAGE").astype(int)

In [11]:
# Check the mapping status
print(tx_master["crime_category_mapped"].value_counts(dropna=False))

crime_category_mapped
OTHER              655072
THEFT              454373
BATTERY            327345
CRIMINAL_DAMAGE    164903
Name: count, dtype: int64


In [12]:
# See which specific offences are included in each category.
for cat in ["THEFT", "BATTERY", "CRIMINAL_DAMAGE", "OTHER"]:
    print(f"\n=== {cat} ===")
    print(sorted(tx_master.loc[tx_master["crime_category_mapped"] == cat, "offense_name"].dropna().unique().tolist()))


=== THEFT ===
['All Other Larceny', 'Pocket-picking', 'Purse-snatching', 'Shoplifting', 'Theft From Building', 'Theft From Coin-Operated Machine or Device', 'Theft From Motor Vehicle', 'Theft of Motor Vehicle Parts or Accessories']

=== BATTERY ===
['Aggravated Assault', 'Simple Assault']

=== CRIMINAL_DAMAGE ===
['Destruction/Damage/Vandalism of Property']

=== OTHER ===
['Animal Cruelty', 'Arson', 'Assisting or Promoting Prostitution', 'Betting/Wagering', 'Bribery', 'Burglary/Breaking & Entering', 'Counterfeiting/Forgery', 'Credit Card/Automated Teller Machine Fraud', 'Criminal Sexual Contact', 'Drug Equipment Violations', 'Drug/Narcotic Violations', 'Embezzlement', 'Extortion/Blackmail', 'False Pretenses/Swindle/Confidence Game', 'Gambling Equipment Violation', 'Hacking/Computer Invasion', 'Human Trafficking, Commercial Sex Acts', 'Human Trafficking, Involuntary Servitude', 'Identity Theft', 'Impersonation', 'Incest', 'Intimidation', 'Justifiable Homicide', 'Kidnapping/Abduction', 

In [13]:
# Build week fields
tx_master["incident_date"] = pd.to_datetime(tx_master["incident_date"], errors="coerce")

iso_cal = tx_master["incident_date"].dt.isocalendar()
tx_master["iso_year"] = iso_cal.year.astype(int)
tx_master["iso_week"] = iso_cal.week.astype(int)

tx_master["year_week"] = (
    tx_master["iso_year"].astype(str) + "-W" +
    tx_master["iso_week"].astype(str).str.zfill(2)
)

print(tx_master[["incident_date", "iso_year", "iso_week", "year_week"]].head(10))
print("Unique year_week:", tx_master["year_week"].nunique())
print("Week range:", tx_master["iso_week"].min(), "to", tx_master["iso_week"].max())

# Aggregate to agency x week
tx_weekly = (
    tx_master
    .groupby(["agency_id", "iso_year", "iso_week", "year_week"], as_index=False)
    .agg(
        total_crimes=("is_total_crime", "sum"),
        count_THEFT=("is_theft", "sum"),
        count_BATTERY=("is_battery", "sum"),
        count_CRIMINAL_DAMAGE=("is_criminal_damage", "sum")
    )
)

print("\nWeekly aggregated shape:", tx_weekly.shape)
print(tx_weekly.head(10))

# Add static agency info back
agency_info = (
    tx_master[["agency_id", "county_name", "msa_name", "population", "agency_type_name"]]
    .drop_duplicates(subset=["agency_id"])
    .copy()
)

tx_weekly = tx_weekly.merge(
    agency_info,
    on="agency_id",
    how="left"
)

print("\nAfter merging agency info:", tx_weekly.shape)
print(tx_weekly[["agency_id", "year_week", "total_crimes", "county_name", "population"]].head(10))

# Create binary labels
tx_weekly["label_total_crime"] = (tx_weekly["total_crimes"] > 0).astype(int)
tx_weekly["label_THEFT"] = (tx_weekly["count_THEFT"] > 0).astype(int)
tx_weekly["label_BATTERY"] = (tx_weekly["count_BATTERY"] > 0).astype(int)
tx_weekly["label_CRIMINAL_DAMAGE"] = (tx_weekly["count_CRIMINAL_DAMAGE"] > 0).astype(int)

print("\nLabel preview:")
print(tx_weekly[[
    "total_crimes", "count_THEFT", "count_BATTERY", "count_CRIMINAL_DAMAGE",
    "label_total_crime", "label_THEFT", "label_BATTERY", "label_CRIMINAL_DAMAGE"
]].head(10))

  incident_date  iso_year  iso_week year_week
0    2024-01-01      2024         1  2024-W01
1    2024-01-01      2024         1  2024-W01
2    2024-01-01      2024         1  2024-W01
3    2024-01-01      2024         1  2024-W01
4    2024-01-02      2024         1  2024-W01
5    2024-01-02      2024         1  2024-W01
6    2024-01-02      2024         1  2024-W01
7    2024-01-04      2024         1  2024-W01
8    2024-01-04      2024         1  2024-W01
9    2024-01-04      2024         1  2024-W01
Unique year_week: 53
Week range: 1 to 52

Weekly aggregated shape: (45782, 8)
   agency_id  iso_year  iso_week year_week  total_crimes  count_THEFT  \
0      18753      2024         1  2024-W01            21            6   
1      18753      2024         2  2024-W02            16            2   
2      18753      2024         3  2024-W03            16            3   
3      18753      2024         4  2024-W04            21            4   
4      18753      2024         5  2024-W05         

In [14]:
# All agencies
all_agencies = sorted(tx_weekly["agency_id"].dropna().unique().tolist())
print("Number of agencies:", len(all_agencies))

# All weeks
all_weeks = (
    tx_weekly[["iso_year", "iso_week", "year_week"]]
    .drop_duplicates()
    .sort_values(["iso_year", "iso_week"])
    .reset_index(drop=True)
)
print("Number of weeks:", all_weeks.shape[0])
print(all_weeks.head())
print(all_weeks.tail())

# Cartesian product agency x week
agency_df = pd.DataFrame({"agency_id": all_agencies})
agency_df["key"] = 1
all_weeks["key"] = 1

tx_full_panel = agency_df.merge(all_weeks, on="key").drop(columns="key")
print("Full panel before merge:", tx_full_panel.shape)

# Merge actual weekly stats back
tx_full_panel = tx_full_panel.merge(
    tx_weekly,
    on=["agency_id", "iso_year", "iso_week", "year_week"],
    how="left"
)
print("Full panel after merge:", tx_full_panel.shape)

# Fill missing counts and labels with 0
count_cols = [
    "total_crimes",
    "count_THEFT",
    "count_BATTERY",
    "count_CRIMINAL_DAMAGE"
]

label_cols = [
    "label_total_crime",
    "label_THEFT",
    "label_BATTERY",
    "label_CRIMINAL_DAMAGE"
]

tx_full_panel[count_cols] = tx_full_panel[count_cols].fillna(0)
tx_full_panel[label_cols] = tx_full_panel[label_cols].fillna(0)

for col in count_cols + label_cols:
    tx_full_panel[col] = tx_full_panel[col].astype(int)

# Re-merge static agency info
tx_full_panel = tx_full_panel.drop(
    columns=["county_name", "msa_name", "population", "agency_type_name"],
    errors="ignore"
)

tx_full_panel = tx_full_panel.merge(
    agency_info,
    on="agency_id",
    how="left"
)

# Check label prevalence
print("\nLabel rates after full panel completion:")
print(tx_full_panel[[
    "label_total_crime",
    "label_THEFT",
    "label_BATTERY",
    "label_CRIMINAL_DAMAGE"
]].mean())

print("\nPreview:")
print(tx_full_panel.head(10))

Number of agencies: 1259
Number of weeks: 53
   iso_year  iso_week year_week
0      2024         1  2024-W01
1      2024         2  2024-W02
2      2024         3  2024-W03
3      2024         4  2024-W04
4      2024         5  2024-W05
    iso_year  iso_week year_week
48      2024        49  2024-W49
49      2024        50  2024-W50
50      2024        51  2024-W51
51      2024        52  2024-W52
52      2025         1  2025-W01
Full panel before merge: (66727, 4)
Full panel after merge: (66727, 16)

Label rates after full panel completion:
label_total_crime        0.686109
label_THEFT              0.448050
label_BATTERY            0.448904
label_CRIMINAL_DAMAGE    0.307027
dtype: float64

Preview:
   agency_id  iso_year  iso_week year_week  total_crimes  count_THEFT  \
0      18753      2024         1  2024-W01            21            6   
1      18753      2024         2  2024-W02            16            2   
2      18753      2024         3  2024-W03            16            3  

In [16]:
import numpy as np

# sort
tx_full_panel = tx_full_panel.sort_values(
    ["agency_id", "iso_year", "iso_week"]
).reset_index(drop=True)

# lag features
lag_steps = [1, 2, 4, 8, 13, 26, 52]

for lag in lag_steps:
    tx_full_panel[f"lag_{lag}w_total"] = (
        tx_full_panel.groupby("agency_id")["total_crimes"].shift(lag)
    )
    tx_full_panel[f"lag_{lag}w_THEFT"] = (
        tx_full_panel.groupby("agency_id")["count_THEFT"].shift(lag)
    )
    tx_full_panel[f"lag_{lag}w_BATTERY"] = (
        tx_full_panel.groupby("agency_id")["count_BATTERY"].shift(lag)
    )
    tx_full_panel[f"lag_{lag}w_CRIMINAL_DAMAGE"] = (
        tx_full_panel.groupby("agency_id")["count_CRIMINAL_DAMAGE"].shift(lag)
    )

# rolling features
rolling_windows = [4, 12, 26]

for window in rolling_windows:
    tx_full_panel[f"roll_{window}w_total"] = (
        tx_full_panel.groupby("agency_id")["total_crimes"]
        .shift(1)
        .rolling(window)
        .mean()
        .reset_index(level=0, drop=True)
    )
    tx_full_panel[f"roll_{window}w_THEFT"] = (
        tx_full_panel.groupby("agency_id")["count_THEFT"]
        .shift(1)
        .rolling(window)
        .mean()
        .reset_index(level=0, drop=True)
    )
    tx_full_panel[f"roll_{window}w_BATTERY"] = (
        tx_full_panel.groupby("agency_id")["count_BATTERY"]
        .shift(1)
        .rolling(window)
        .mean()
        .reset_index(level=0, drop=True)
    )
    tx_full_panel[f"roll_{window}w_CRIMINAL_DAMAGE"] = (
        tx_full_panel.groupby("agency_id")["count_CRIMINAL_DAMAGE"]
        .shift(1)
        .rolling(window)
        .mean()
        .reset_index(level=0, drop=True)
    )

# time features
tx_full_panel["sin_week"] = np.sin(2 * np.pi * tx_full_panel["iso_week"] / 52)
tx_full_panel["cos_week"] = np.cos(2 * np.pi * tx_full_panel["iso_week"] / 52)
tx_full_panel["year_trend"] = tx_full_panel["iso_year"] - tx_full_panel["iso_year"].min()

# check missing
lag_cols = [c for c in tx_full_panel.columns if c.startswith("lag_")]
roll_cols = [c for c in tx_full_panel.columns if c.startswith("roll_")]

print(tx_full_panel[lag_cols + roll_cols].isna().sum().sort_values(ascending=False).head(20))

# fill missing for now
tx_full_panel[lag_cols + roll_cols] = tx_full_panel[lag_cols + roll_cols].fillna(0)

print(tx_full_panel.head(10))

lag_52w_CRIMINAL_DAMAGE     65468
lag_52w_BATTERY             65468
lag_52w_THEFT               65468
lag_52w_total               65468
lag_26w_total               32734
lag_26w_THEFT               32734
roll_26w_BATTERY            32734
roll_26w_THEFT              32734
roll_26w_total              32734
lag_26w_CRIMINAL_DAMAGE     32734
lag_26w_BATTERY             32734
roll_26w_CRIMINAL_DAMAGE    32734
lag_13w_total               16367
lag_13w_THEFT               16367
lag_13w_BATTERY             16367
lag_13w_CRIMINAL_DAMAGE     16367
roll_12w_total              15108
roll_12w_THEFT              15108
roll_12w_BATTERY            15108
roll_12w_CRIMINAL_DAMAGE    15108
dtype: int64
   agency_id  iso_year  iso_week year_week  total_crimes  count_THEFT  \
0      18753      2024         1  2024-W01            21            6   
1      18753      2024         2  2024-W02            16            2   
2      18753      2024         3  2024-W03            16            3   
3      18753   

In [17]:
# Define column groups
id_cols = ["agency_id", "iso_year", "iso_week", "year_week"]

target_count_cols = [
    "total_crimes",
    "count_THEFT",
    "count_BATTERY",
    "count_CRIMINAL_DAMAGE"
]

label_cols = [
    "label_total_crime",
    "label_THEFT",
    "label_BATTERY",
    "label_CRIMINAL_DAMAGE"
]

meta_cols = [
    "county_name",
    "msa_name",
    "population",
    "agency_type_name"
]

lag_cols = [c for c in tx_full_panel.columns if c.startswith("lag_")]
roll_cols = [c for c in tx_full_panel.columns if c.startswith("roll_")]

feature_cols = lag_cols + roll_cols + ["sin_week", "cos_week", "population", "year_trend"]

# Check missing in feature columns
print(tx_full_panel[feature_cols].isna().sum().sort_values(ascending=False).head(20))

# Fill any remaining numeric missing values
tx_full_panel["population"] = tx_full_panel["population"].fillna(0)

# year_trend usually all 0 here, keep for now
tx_full_panel[feature_cols] = tx_full_panel[feature_cols].fillna(0)

# Build final standardized external-test table
tx_external_final = tx_full_panel[
    id_cols + target_count_cols + label_cols + meta_cols + feature_cols
].copy()

print(tx_external_final.shape)
print(tx_external_final.head(10))

# Build model-ready X view (NO leakage columns)
X_tx = tx_external_final[feature_cols].copy()

print("X_tx shape:", X_tx.shape)
print(X_tx.head(10))

# Save
tx_external_final.to_csv("tx_external_test_panel_final.csv", index=False)
print("Saved: tx_external_test_panel_final.csv")

population                  106
lag_1w_total                  0
roll_12w_total                0
lag_52w_total                 0
lag_52w_THEFT                 0
lag_52w_BATTERY               0
lag_52w_CRIMINAL_DAMAGE       0
roll_4w_total                 0
roll_4w_THEFT                 0
roll_4w_BATTERY               0
roll_4w_CRIMINAL_DAMAGE       0
roll_12w_THEFT                0
lag_1w_THEFT                  0
roll_12w_BATTERY              0
roll_12w_CRIMINAL_DAMAGE      0
roll_26w_total                0
roll_26w_THEFT                0
roll_26w_BATTERY              0
roll_26w_CRIMINAL_DAMAGE      0
sin_week                      0
dtype: int64
(66727, 60)
   agency_id  iso_year  iso_week year_week  total_crimes  count_THEFT  \
0      18753      2024         1  2024-W01            21            6   
1      18753      2024         2  2024-W02            16            2   
2      18753      2024         3  2024-W03            16            3   
3      18753      2024         4  2024-W04 

In [19]:
from collections import Counter

# AI GPT, help me check the current dataset and debug it.
# Check duplicated features
dup_cols = [col for col, count in Counter(feature_cols).items() if count > 1]
print("Duplicated feature columns:", dup_cols)

# Remove duplicates while preserving order
feature_cols = list(dict.fromkeys(feature_cols))

# Optionally drop year_trend because Texas is essentially single-year
feature_cols = [c for c in feature_cols if c != "year_trend"]

# Rebuild X_tx
X_tx = tx_external_final[feature_cols].copy()

print("Final X_tx shape:", X_tx.shape)
print("First 20 feature cols:", feature_cols[:20])
print("Total feature count:", len(feature_cols))

Duplicated feature columns: []
Final X_tx shape: (66727, 44)
First 20 feature cols: ['lag_1w_total', 'lag_1w_THEFT', 'lag_1w_BATTERY', 'lag_1w_CRIMINAL_DAMAGE', 'lag_2w_total', 'lag_2w_THEFT', 'lag_2w_BATTERY', 'lag_2w_CRIMINAL_DAMAGE', 'lag_4w_total', 'lag_4w_THEFT', 'lag_4w_BATTERY', 'lag_4w_CRIMINAL_DAMAGE', 'lag_8w_total', 'lag_8w_THEFT', 'lag_8w_BATTERY', 'lag_8w_CRIMINAL_DAMAGE', 'lag_13w_total', 'lag_13w_THEFT', 'lag_13w_BATTERY', 'lag_13w_CRIMINAL_DAMAGE']
Total feature count: 43


In [20]:
print(tx_external_final.columns[tx_external_final.columns.duplicated()].tolist())

['population']


In [21]:
tx_external_final = tx_external_final.loc[:, ~tx_external_final.columns.duplicated()].copy()
X_tx = tx_external_final[feature_cols].copy()

print("Final X_tx shape:", X_tx.shape)
print("Total feature count:", len(feature_cols))

Final X_tx shape: (66727, 43)
Total feature count: 43


In [22]:
# Standardised External Measurement Instrument
tx_external_final = tx_external_final.loc[:, ~tx_external_final.columns.duplicated()].copy()
tx_external_final.to_csv("tx_external_test_panel_final_clean.csv", index=False)

In [23]:
# Pure model feature matrix
X_tx.to_csv("tx_external_X_shared_43cols.csv", index=False)